# Huấn Luyện Deep Neural Network Khảo Sát Kiến Trúc

Notebook này thiết lập các kiến trúc Deep Neural Network (Mạng nơ-ron sâu) khác nhau sử dụng `MLPClassifier`. Mục tiêu là thực nghiệm xem việc tăng chiều sâu (depth) hoặc chiều rộng (width) của mạng nơ-ron có giúp phát hiện mã nguồn lỗi tốt hơn các thuật toán học máy truyền thống hay không.


In [1]:
import sys
import os
import glob
import numpy as np
import pandas as pd

sys.path.append(os.path.abspath('..'))

from src.data_loader import load_arff_dataset
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import warnings
from IPython.display import display

warnings.filterwarnings('ignore')

In [2]:
from sklearn.neural_network import MLPClassifier

def get_models(random_state=42):
    # Chúng ta sẽ sử dụng tham số early_stopping=True để tránh overfitting trên các tập nhỏ
    # max_iter=1000 đảm bảo mạng nơ ron có đủ thời gian hội tụ
    return {
        'DNN_Shallow': MLPClassifier(
            hidden_layer_sizes=(64,),
            activation='relu', solver='adam', early_stopping=True, max_iter=1000, random_state=random_state
        ),
        'DNN_Medium': MLPClassifier(
            hidden_layer_sizes=(128, 64),
            activation='relu', solver='adam', early_stopping=True, max_iter=1000, random_state=random_state
        ),
        'DNN_Deep': MLPClassifier(
            hidden_layer_sizes=(128, 64, 32),
            activation='relu', solver='adam', early_stopping=True, max_iter=1000, random_state=random_state
        ),
        'DNN_Wide': MLPClassifier(
            hidden_layer_sizes=(256, 128),
            activation='relu', solver='adam', early_stopping=True, max_iter=1000, random_state=random_state
        )
    }

In [3]:
def run_experiment_on_dataset(df, target_col='Defective', random_state=42):
    df = df.dropna()
    X = df.drop(columns=[target_col]).values
    y = df[target_col].map({'Defective': 1, 'Clean': 0}).values
    
    kf = KFold(n_splits=10, shuffle=True, random_state=random_state)
    models_dict = get_models()
    
    results_accumulator = {
        name: {'Accuracy': [], 'Precision': [], 'Recall': [], 'F-Score': [], 'ROC-AUC': []}
        for name in models_dict.keys()
    }
    
    for fold_idx, (train_idx, test_idx) in enumerate(kf.split(X, y)):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        for model_name, model in models_dict.items():
            pipeline = ImbPipeline([
                ('scaler', StandardScaler()),
                ('smote', SMOTE(random_state=random_state)),
                ('model', model)
            ])
            
            try:
                pipeline.fit(X_train, y_train)
                predictions = pipeline.predict(X_test)
                if hasattr(pipeline.named_steps['model'], "predict_proba"):
                    probabilities = pipeline.predict_proba(X_test)[:, 1]
                else:
                    probabilities = pipeline.decision_function(X_test)
                
                acc = accuracy_score(y_test, predictions)
                prec = precision_score(y_test, predictions, average='weighted', zero_division=0)
                rec = recall_score(y_test, predictions, average='weighted', zero_division=0)
                f1 = f1_score(y_test, predictions, average='weighted', zero_division=0)
                try:
                    auc = roc_auc_score(y_test, probabilities, average='weighted')
                except ValueError:
                    auc = np.nan
                    
                results_accumulator[model_name]['Accuracy'].append(acc)
                results_accumulator[model_name]['Precision'].append(prec)
                results_accumulator[model_name]['Recall'].append(rec)
                results_accumulator[model_name]['F-Score'].append(f1)
                results_accumulator[model_name]['ROC-AUC'].append(auc)
            except Exception as e:
                pass
                
    final_report = {}
    for model_name, metrics in results_accumulator.items():
        final_report[model_name] = {
            'Accuracy': np.round(np.mean(metrics['Accuracy']), 3) if metrics['Accuracy'] else 0.0,
            'Precision': np.round(np.mean(metrics['Precision']), 3) if metrics['Precision'] else 0.0,
            'Recall': np.round(np.mean(metrics['Recall']), 3) if metrics['Recall'] else 0.0,
            'F-Score': np.round(np.mean(metrics['F-Score']), 3) if metrics['F-Score'] else 0.0,
            'ROC-AUC': np.round(np.nanmean(metrics['ROC-AUC']), 3) if metrics['ROC-AUC'] else 0.0
        }
    return final_report

In [4]:
dataset_dir = '../datasets'
arff_files = glob.glob(os.path.join(dataset_dir, '*.arff'))

all_results = []

print(f"Tìm thấy {len(arff_files)} tập dữ liệu. Bắt đầu huấn luyện mạng nơ-ron...")

for file_path in sorted(arff_files):
    ds_name = os.path.splitext(os.path.basename(file_path))[0]
    print(f"\n>>> Đang xử lý tập dữ liệu: {ds_name}...")
    df, target_col = load_arff_dataset(file_path)
    report = run_experiment_on_dataset(df, target_col)
    
    for model_name, metrics in report.items():
        row = {'Dataset': ds_name, 'Model': model_name}
        row.update(metrics)
        all_results.append(row)
        
results_df = pd.DataFrame(all_results)
results_df = results_df[['Dataset', 'Model', 'Accuracy', 'Precision', 'Recall', 'F-Score', 'ROC-AUC']]

print("\n================ HOÀN TẤT THỰC NGHIỆM DNN ================")
display(results_df)

# Lưu báo cáo
output_file = 'report_DNN.csv'
results_df.to_csv(output_file, index=False)
print(f"\nKết quả đã được lưu thành công vào: {output_file}")

Tìm thấy 10 tập dữ liệu. Bắt đầu huấn luyện mạng nơ-ron...

>>> Đang xử lý tập dữ liệu: JM1...

>>> Đang xử lý tập dữ liệu: KC3...

>>> Đang xử lý tập dữ liệu: MC1...

>>> Đang xử lý tập dữ liệu: MC2...

>>> Đang xử lý tập dữ liệu: MW1...

>>> Đang xử lý tập dữ liệu: PC1...

>>> Đang xử lý tập dữ liệu: PC2...

>>> Đang xử lý tập dữ liệu: PC3...

>>> Đang xử lý tập dữ liệu: PC4...

>>> Đang xử lý tập dữ liệu: PC5...

================ HOÀN TẤT THỰC NGHIỆM DNN ================


,Dataset,Model,Accuracy,Precision,Recall,F-Score,ROC-AUC
0,JM1,DNN_Shallow,0.686,0.756,0.686,0.710,0.696
1,JM1,DNN_Medium,0.693,0.753,0.693,0.713,0.669
2,JM1,DNN_Deep,0.694,0.743,0.694,0.712,0.639
3,JM1,DNN_Wide,0.695,0.736,0.695,0.711,0.635
4,KC3,DNN_Shallow,0.711,0.801,0.711,0.735,0.707
5,KC3,DNN_Medium,0.666,0.759,0.666,0.692,0.709
6,KC3,DNN_Deep,0.697,0.816,0.697,0.723,0.759
7,KC3,DNN_Wide,0.728,0.789,0.728,0.745,0.751
8,MC1,DNN_Shallow,0.943,0.976,0.943,0.957,0.776
9,MC1,DNN_Medium,0.968,0.974,0.968,0.971,0.732



Kết quả đã được lưu thành công vào: report_DNN.csv
